# SEEDS / ARISE — Land cover × Census block groups (12 counties)
1. Loads the 12 NAIP RF land cover assets (1 m) and NLCD 2021 (30 m)
2. Reads `Us_AK_merged_block_groups_with_pop.shp` from Drive, clipped to the 12 counties
3. Computes hectares and % of each NLCD class per block group from **both** layers
   (Colab sends one request per block group; nothing is exported or saved to Earth Engine)
4. Joins to every census attribute → block-group table, plus tract and county roll-ups
5. Map: NAIP RF, NLCD, block groups by population density

In [ ]:
!pip install -q geemap pyogrio gdown
from google.colab import drive
drive.mount('/content/drive')

import os, glob, time, json
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import box
import ee, geemap

PROJECT = 'servir-sco-assets'
ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
# ============================== SETTINGS ====================================
SHP_NAME   = 'Us_AK_merged_block_groups_with_pop.shp'
SHP_PATH   = None   # set the full path if auto-search misses it, e.g.
                    # '/content/drive/MyDrive/<folder>/Us_AK_merged_block_groups_with_pop.shp'
DRIVE_FOLDER_ID = '1lAWA0UugMbXTZo3Jlc8pvWH1Q1yER6_l'   # fallback download with gdown
POP_COL    = 'U7H001'   # NHGIS 2020 total population (None = auto-detect a column containing 'pop')

RF_ASSETS = [f'projects/{PROJECT}/assets/naip_rf_landcover/{k}' for k in [
    'TE_Bledsoe', 'TE_Grundy', 'TE_VanBuren', 'TE_DeKalb', 'TE_Warren', 'TE_Sequatchie',
    'AL_Jackson', 'AL_Marshall', 'TE_Coffee', 'TE_Marion', 'TE_Franklin', 'AL_Madison']]

# county FIPS (state + county) for the 12 counties of interest
COUNTY_FIPS = {
    '47007': 'Bledsoe, TN',  '47061': 'Grundy, TN',  '47175': 'Van Buren, TN', '47041': 'DeKalb, TN',
    '47177': 'Warren, TN',   '47153': 'Sequatchie, TN', '47031': 'Coffee, TN', '47115': 'Marion, TN',
    '47051': 'Franklin, TN', '01071': 'Jackson, AL', '01095': 'Marshall, AL', '01089': 'Madison, AL'}

NAIP_SCALE = 1      # m for the RF zonal stats (2 or 5 = much faster, small area error)
NLCD_SCALE = 30
WORKERS    = 16     # parallel Earth Engine requests from Colab
OUT_DIR    = '/content/drive/MyDrive/ARSIE_MAP_Creation/landcover_census'
os.makedirs(OUT_DIR, exist_ok=True)

CODES = [11, 12, 21, 22, 23, 24, 31, 41, 42, 43, 52, 71, 81, 82, 90, 95]
NAMES = ['Open Water', 'Perennial Ice/Snow', 'Developed, Open Space', 'Developed, Low',
         'Developed, Medium', 'Developed, High', 'Barren Land', 'Deciduous Forest',
         'Evergreen Forest', 'Mixed Forest', 'Shrub/Scrub', 'Grassland/Herbaceous',
         'Pasture/Hay', 'Cultivated Crops', 'Woody Wetlands', 'Emergent Herbaceous Wetlands']
PALETTE = ['466b9f', 'd1def8', 'dec5c5', 'd99282', 'eb0000', 'ab0000', 'b3ac9f', '68ab5f',
           '1c5f2c', 'b5c58f', 'ccb879', 'dfdfc2', 'dcd939', 'ab6c28', 'b8d9eb', '6c9fb8']
GROUPS = {'water': [11], 'developed': [21, 22, 23, 24], 'barren': [31], 'forest': [41, 42, 43],
          'shrub_grass': [52, 71], 'agriculture': [81, 82], 'wetlands': [90, 95]}

In [ ]:
# ============================== LAYERS (EE) =================================
rf = ee.ImageCollection(RF_ASSETS).mosaic().rename('landcover')
nlcd = ee.Image('USGS/NLCD_RELEASES/2021_REL/NLCD/2021').select('landcover')

tiger = ee.FeatureCollection('TIGER/2018/Counties') \
          .filter(ee.Filter.inList('GEOID', list(COUNTY_FIPS)))
counties_gdf = geemap.ee_to_gdf(tiger)[['GEOID', 'NAME', 'STATEFP', 'geometry']].to_crs(4326)
print(len(counties_gdf), 'counties:', sorted(counties_gdf.NAME))
for a in RF_ASSETS:      # quick existence check
    try: ee.data.getAsset(a)
    except ee.EEException: print('MISSING asset:', a)

In [ ]:
# ============================== CENSUS BLOCK GROUPS =========================
def find_shp():
    if SHP_PATH and os.path.exists(SHP_PATH):
        return SHP_PATH
    for root in ['/content/drive/MyDrive', '/content/drive/Shareddrives', '/content/drive/.shortcut-targets-by-id']:
        hits = glob.glob(f'{root}/**/{SHP_NAME}', recursive=True)
        if hits: return hits[0]
    # fallback: download the shared folder (needs link sharing on)
    import gdown
    gdown.download_folder(id=DRIVE_FOLDER_ID, output='/content/census_bg', quiet=False)
    hits = glob.glob(f'/content/census_bg/**/{SHP_NAME}', recursive=True)
    if hits: return hits[0]
    raise FileNotFoundError(f'{SHP_NAME} not found: set SHP_PATH')

shp = find_shp(); print('Reading', shp)
aoi_box = gpd.GeoSeries([box(*counties_gdf.total_bounds).buffer(0.02)], crs=4326)
bg = gpd.read_file(shp, bbox=aoi_box, engine='pyogrio')       # only rows near the AOI
print('Rows in AOI bbox:', len(bg), '| CRS:', bg.crs)
print('Columns:', list(bg.columns))

# ---- identify GEOID / FIPS; fall back to spatial filter
geoid_col = next((c for c in bg.columns if c.upper() in ('GEOID', 'GEOID20', 'GEOID10', 'BG_GEOID', 'FIPS')), None)
if geoid_col:
    bg['GEOID_BG'] = bg[geoid_col].astype(str).str.replace(r'\D', '', regex=True).str[-12:].str.zfill(12)
    bg = bg[bg['GEOID_BG'].str[:5].isin(COUNTY_FIPS)].copy()
else:
    # no GEOID column: build it from FIPS parts if present, else spatial join to counties
    parts = ['STATEFP', 'COUNTYFP', 'TRACTCE', 'BLKGRPCE']
    if all(c in bg.columns for c in parts):
        bg['GEOID_BG'] = (bg.STATEFP.astype(str).str.zfill(2) + bg.COUNTYFP.astype(str).str.zfill(3)
                          + bg.TRACTCE.astype(str).str.zfill(6) + bg.BLKGRPCE.astype(str))
        bg = bg[bg['GEOID_BG'].str[:5].isin(COUNTY_FIPS)].copy()
    else:
        cent = gpd.GeoDataFrame(geometry=bg.to_crs(5070).representative_point().to_crs(4326), crs=4326)
        j = gpd.sjoin(cent, counties_gdf, predicate='within')
        bg = bg.loc[j.index].copy()
        bg['GEOID_BG'] = j['GEOID'].values + pd.Series(range(len(bg))).astype(str).str.zfill(7).values
        print('WARNING: no GEOID/FIPS columns; tract/BG ids are synthetic — set the id column')
bg['STATE_FIPS']  = bg.GEOID_BG.str[:2]
bg['COUNTY_FIPS'] = bg.GEOID_BG.str[:5]
bg['COUNTY']      = bg.COUNTY_FIPS.map(COUNTY_FIPS)
bg['TRACT_GEOID'] = bg.GEOID_BG.str[:11]
bg['TRACT']       = bg.GEOID_BG.str[5:11].str[:4].str.lstrip('0') + '.' + bg.GEOID_BG.str[9:11]
bg['BLOCK_GROUP'] = bg.GEOID_BG.str[11]
bg = bg.to_crs(4326).drop(columns=[c for c in ['index_righ'] if c in bg.columns])

POP = POP_COL if POP_COL in bg.columns else next((c for c in bg.columns if 'pop' in c.lower()), None)
print('Population column:', POP)
bg['area_km2'] = bg.to_crs(5070).area / 1e6
if 'ALAND' in bg.columns:
    bg['land_km2'] = pd.to_numeric(bg['ALAND'], errors='coerce') / 1e6
else:
    bg['land_km2'] = bg['area_km2']
if POP:
    bg[POP] = pd.to_numeric(bg[POP], errors='coerce')
    bg['pop_density_km2'] = bg[POP] / bg['land_km2']
print(bg.groupby('COUNTY').size().rename('block_groups'))

In [ ]:
# ============================== ZONAL LAND COVER (from Colab) ===============
from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely.geometry import mapping

CKPT = f'{OUT_DIR}/_zonal_checkpoint.csv'      # lets a re-run resume where it stopped


def class_ha(img, geom, scale):
    area = ee.Image.pixelArea().divide(1e4).addBands(img)          # ha per pixel + class
    return area.reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
                             geometry=geom, scale=scale, maxPixels=1e13, tileScale=16)


def bg_stats(geoid, shp_geom, prefix, img, scale):
    geom = ee.Geometry(mapping(shp_geom), 'EPSG:4326', False)
    err = None
    for s in (scale, scale * 2, scale * 5):          # coarser retry if a big block group times out
        try:
            out = class_ha(img, geom, s).getInfo()
            d = {'GEOID_BG': geoid, 'layer': prefix, 'scale_m': s, **{f'ha_{c}': 0.0 for c in CODES}}
            for g in out.get('groups', []):
                d[f'ha_{int(g["class"])}'] = g['sum']
            return d
        except Exception as e:
            err = e
    return {'GEOID_BG': geoid, 'layer': prefix, 'error': str(err)[:200]}


geoms = bg.assign(geometry=bg.to_crs(5070).simplify(1).to_crs(4326).geometry)   # 1 m tolerance
done = pd.read_csv(CKPT, dtype={'GEOID_BG': str}) if os.path.exists(CKPT) else pd.DataFrame()
if len(done) and 'error' in done:
    done = done[done['error'].isna()]
have = set(zip(done.get('GEOID_BG', []), done.get('layer', [])))
jobs = [(r.GEOID_BG, r.geometry, p, img, sc) for r in geoms.itertuples()
        for p, img, sc in [('nlcd', nlcd, NLCD_SCALE), ('naip', rf, NAIP_SCALE)]
        if (r.GEOID_BG, p) not in have]
print(f'{len(jobs)} requests to run ({len(have)} already in checkpoint)')

rows, t0 = list(done.to_dict('records')), time.time()
with ThreadPoolExecutor(WORKERS) as ex:
    futs = [ex.submit(bg_stats, *j) for j in jobs]
    for i, f in enumerate(as_completed(futs), 1):
        rows.append(f.result())
        if i % 50 == 0 or i == len(futs):
            pd.DataFrame(rows).to_csv(CKPT, index=False)
            print(f'  {i}/{len(futs)}  ({(time.time() - t0) / 60:.1f} min)')

zon = pd.DataFrame(rows)
if 'error' in zon and zon['error'].notna().any():
    print('Failed (re-run this cell to retry):'); display(zon[zon['error'].notna()][['GEOID_BG', 'layer', 'error']])
zon = zon[zon.get('error', pd.Series(index=zon.index)).isna()]
print('scale used (m) per layer:'); print(zon.groupby('layer')['scale_m'].value_counts())

# ============================== BUILD TABLES ================================
def layer_df(prefix):
    d = zon[zon.layer == prefix].drop(columns=['layer'] + [c for c in ['error'] if c in zon])
    return d.rename(columns={**{f'ha_{c}': f'{prefix}_ha_{c}' for c in CODES}, 'scale_m': f'{prefix}_scale_m'})


naip_df, nlcd_df = layer_df('naip'), layer_df('nlcd')


def add_pct(df, prefix):
    ha = [f'{prefix}_ha_{c}' for c in CODES]
    df[f'{prefix}_ha_total'] = df[ha].sum(axis=1)
    for c in CODES:
        df[f'{prefix}_pct_{c}'] = 100 * df[f'{prefix}_ha_{c}'] / df[f'{prefix}_ha_total']
    for g, cs in GROUPS.items():
        df[f'{prefix}_{g}_ha'] = df[[f'{prefix}_ha_{c}' for c in cs]].sum(axis=1)
        df[f'{prefix}_{g}_pct'] = 100 * df[f'{prefix}_{g}_ha'] / df[f'{prefix}_ha_total']
    return df


lc = add_pct(naip_df, 'naip').merge(add_pct(nlcd_df, 'nlcd'), on='GEOID_BG', how='outer')
for g in GROUPS:   # RF (1 m) minus NLCD (30 m), percentage points
    lc[f'diff_{g}_pct'] = lc[f'naip_{g}_pct'] - lc[f'nlcd_{g}_pct']
if POP:
    lc = lc.merge(bg[['GEOID_BG', POP]], on='GEOID_BG')
    for p in ('naip', 'nlcd'):
        lc[f'{p}_forest_ha_per_1000ppl'] = 1000 * lc[f'{p}_forest_ha'] / lc[POP].replace(0, np.nan)
        lc[f'{p}_developed_m2_per_person'] = 1e4 * lc[f'{p}_developed_ha'] / lc[POP].replace(0, np.nan)
    lc = lc.drop(columns=POP)

id_cols = ['COUNTY', 'COUNTY_FIPS', 'TRACT', 'TRACT_GEOID', 'BLOCK_GROUP', 'GEOID_BG']
census_cols = [c for c in bg.columns if c not in id_cols + ['geometry']]
bg_lc = bg[id_cols + census_cols + ['geometry']].merge(lc, on='GEOID_BG', how='left')
table = pd.DataFrame(bg_lc.drop(columns='geometry')).round(3)


# roll-ups: sum population + hectares, recompute %
def rollup(by):
    num = [c for c in table.columns if (c.endswith('_ha') or '_ha_' in c) and 'per' not in c and 'scale' not in c] + ([POP] if POP else []) \
          + ['area_km2', 'land_km2']
    r = table.groupby(by)[num].sum(min_count=1)
    r['block_groups'] = table.groupby(by).size()
    for p in ('naip', 'nlcd'):
        for c in CODES: r[f'{p}_pct_{c}'] = 100 * r[f'{p}_ha_{c}'] / r[f'{p}_ha_total']
        for g in GROUPS: r[f'{p}_{g}_pct'] = 100 * r[f'{p}_{g}_ha'] / r[f'{p}_ha_total']
    for g in GROUPS: r[f'diff_{g}_pct'] = r[f'naip_{g}_pct'] - r[f'nlcd_{g}_pct']
    if POP: r['pop_density_km2'] = r[POP] / r['land_km2']
    return r.reset_index().round(3)


tract_tbl = rollup(['COUNTY', 'COUNTY_FIPS', 'TRACT_GEOID', 'TRACT'])
county_tbl = rollup(['COUNTY', 'COUNTY_FIPS'])

table.to_csv(f'{OUT_DIR}/blockgroup_landcover_census.csv', index=False)
tract_tbl.to_csv(f'{OUT_DIR}/tract_landcover_census.csv', index=False)
county_tbl.to_csv(f'{OUT_DIR}/county_landcover_census.csv', index=False)
bg_lc.to_file(f'{OUT_DIR}/blockgroup_landcover_census.gpkg', driver='GPKG')

# legend for column codes
pd.DataFrame({'code': CODES, 'name': NAMES}).to_csv(f'{OUT_DIR}/nlcd_class_codes.csv', index=False)
print('Saved to', OUT_DIR)

show = ['COUNTY', 'block_groups'] + ([POP, 'pop_density_km2'] if POP else []) + \
       [f'{p}_{g}_pct' for g in ('forest', 'developed', 'agriculture', 'wetlands') for p in ('naip', 'nlcd')]
display(county_tbl[show])
display(table[id_cols + ([POP] if POP else []) + ['naip_forest_pct', 'nlcd_forest_pct',
                                                   'naip_developed_pct', 'nlcd_developed_pct']].head(20))

In [ ]:
# ============================== MAP =========================================
pal96 = ['000000'] * 96
for c, p in zip(CODES, PALETTE): pal96[c] = p
lc_vis = {'min': 0, 'max': 95, 'palette': pal96}

bg_ee = geemap.gdf_to_ee(bg_lc[['GEOID_BG', 'pop_density_km2', 'naip_forest_pct', 'geometry']]
                         .assign(geometry=lambda d: d.to_crs(5070).simplify(5).to_crs(4326))
                         .fillna(0)) if POP else geemap.gdf_to_ee(bg[['GEOID_BG', 'geometry']])

Map = geemap.Map(center=[35.0, -86.0], zoom=9)
Map.add_basemap('SATELLITE')
Map.addLayer(nlcd.clip(tiger.geometry()), lc_vis, 'NLCD 2021 (30 m)', False)
Map.addLayer(rf, lc_vis, 'NAIP RF land cover (1 m)')
if POP:
    dens = ee.Image().float().paint(bg_ee, 'pop_density_km2')
    Map.addLayer(dens.log10(), {'min': 0, 'max': 3.5,
                 'palette': ['fff5eb', 'fdbe85', 'fd8d3c', 'd94701', '8c2d04']},
                 'Population density (log10 per km²)', False, 0.7)
    Map.addLayer(ee.Image().float().paint(bg_ee, 'naip_forest_pct'),
                 {'min': 0, 'max': 100, 'palette': ['f7fcf5', '74c476', '00441b']},
                 'Forest % per block group (RF)', False, 0.7)
Map.addLayer(bg_ee.style(color='333333', fillColor='00000000', width=0.6), {}, 'Block groups')
Map.addLayer(tiger.style(color='000000', fillColor='00000000', width=2), {}, 'Counties')
Map.add_legend(title='NLCD classes', legend_dict={f'{c} {n}': p for c, n, p in zip(CODES, NAMES, PALETTE)})
Map